# Build a Baseline Qodec & Instruction Set

<div style="border-left:4px solid #6d3f8c;background:#6d3f8c1a;border-radius:4px;margin:1em 0;"><div style="background:#6d3f8c;color:#ffffff;padding:0.35em 0.8em;font-weight:600;"><span style="display:inline-block;width:1.15em;height:1.15em;line-height:1.15em;border-radius:50%;background:#ffffff;color:#6d3f8c;text-align:center;font-weight:700;margin-right:0.5em;">!</span>Chapter focus</div>

<div style="padding:0.1em 1em;">

How does `ec.build_qodec` synthesize a two-layer QEC protocol, and how are logical instructions specified via instruction actions (`qodec.actions`)?

</div>

</div>

## Learning objectives

After completing this unit, you will be able to:

- Generate a baseline two-layer `qodec.Qodec` protocol from a `qodec.Code` using `ec.build_qodec(code, strategy='bare-css/v1')`.
- Inspect Layer 0 (`InstructionSet`, `codes`, `gadgets`) and Layer 1 (`physical` ISA).
- Extend the physical layer with neutral-atom primitives (`CZ`, `SWAP`) and inspect `.qodec` YAML serialization.

## Before you begin

This course requires `qdk[ec]>=1.33.0`, `qodec>=0.2.0`, `paulimer`, `binar`, and `ipykernel`. Run the cell below to verify your active Python kernel and environment.

In [ ]:
# If packages are missing, first select a dedicated Python environment/kernel,
# then uncomment the next line and run this cell again.
# %pip install -r ../requirements.txt

from _unit import check_env

check_env()

## From a Static Code to an Operational Protocol (`qodec.Qodec`)

A `qodec.Code` tells us *where* information lives, but not *how* to prepare, transform, syndrome-extract, or measure it. In `qodec`, a complete fault-tolerant protocol is represented by a **`Qodec`** object composed of an ordered list of **layers** (`protocol.layers`):
- **Layer 0 (Logical Layer):** Defines the logical instruction set exposed to high-level quantum programs (`prepare_z_all`, `prepare_x_all`, `measure_z_all`, `measure_x_all`, `cx_all`, `syndrome`), the block code (`C4`), and the **gadgets** that implement each logical instruction using Layer-1 operations.
- **Layer 1 (Physical / Stim Layer):** Defines the target hardware or simulator primitives (`R`, `H`, `CX`, `CZ`, `M`) operating on single physical qubits (`Q`).

Each **Instruction** in `layer.instruction_set.instructions` specifies its operational semantics algebraically via **instruction actions** (`instr.action`), observable readouts, and optional rejection flags (`instr.flags`):

| Logical Semantics | Python `qodec.actions` API | `.qodec` YAML Action Syntax | `C4` Example (`protocol.layers[0]`) |
| :--- | :--- | :--- | :--- |
| **State Preparation** | `qodec.actions.Stabilize(["Z_0", "Z_1"])` | `action: [stabilize: [Z_0, Z_1]]` | `prepare_z_all`: prepares $+1$ eigenstate of $\bar{Z}_0, \bar{Z}_1$ ($|00\rangle_L$) |
| **Clifford Unitary** | `qodec.actions.Clifford({"X_0": "X_0 X_2", ...})` | `action: [clifford: {X_0: X_0 X_2, ...}]` | `cx_all`: maps input logical Paulis on two blocks to output Paulis |
| **Pauli Operator** | `qodec.actions.Pauli("X_0")` | `action: [pauli: X_0]` | Logical bit/phase flip up to sign |
| **Measurement** | `qodec.actions.Observe(["Z_0", "Z_1"])` | `action: [observe: [Z_0, Z_1]]` | `measure_z_all`: destructively measures $\bar{Z}_0$ and $\bar{Z}_1$ (`observe_count = 2`) |
| **Syndrome Extraction** | `action=[]` (identity on logicals) | `in: [C4]`, `out: [C4]` | `syndrome`: implements logical identity while refreshing block stabilizer checks |

## Synthesizing a Baseline Protocol with `ec.build_qodec`

Rather than writing hundreds of lines of `.qodec` YAML by hand, `ec.build_qodec(c4, strategy="bare-css/v1")` automatically synthesizes a complete 2-layer protocol for any CSS code.

In [2]:
import qodec
import qdk.ec as ec
from _unit import simulation

# Synthesize a Baseline 2-Layer Protocol with `ec.build_qodec`
from _unit import simulation
simulation.ensure_course_state(globals(), stage="code")

# First, see what happens if we ask `bare-css/v1` for strict fault-distance compliance on C4:
try:
    ec.build_qodec(c4, strategy="bare-css/v1", strict=True)
except Exception as exc:
    print(f"[strict=True guardrail triggered!] {type(exc).__name__}: {exc}\n")

# Now build with strict=False so we can inspect the baseline protocol and upgrade its gadgets:
protocol = ec.build_qodec(c4, strategy="bare-css/v1", strict=False)
gadgets = protocol.layers[0].gadgets

print(f"Protocol has {len(protocol.layers)} layers:")
for idx, layer in enumerate(protocol.layers):
    instr_names = list(layer.instruction_set.instructions.keys())
    code_names = list(layer.codes.keys())
    gadget_names = list(layer.gadgets.keys())
    print(f"  Layer {idx}: instructions={instr_names}, codes={code_names}, gadgets={gadget_names}")

[strict=True guardrail triggered!] ValueError: could not build 'h_all' for code 'C4': verification ActionMismatch: logical action differs between declared and realized



Protocol has 2 layers:
  Layer 0: instructions=['prepare_z_all', 'prepare_x_all', 'syndrome', 'measure_z_all', 'measure_x_all', 'cx_all'], codes=['C4'], gadgets=['cx_all', 'measure_x_all', 'measure_z_all', 'prepare_x_all', 'prepare_z_all', 'syndrome']
  Layer 1: instructions=['R', 'H', 'CX', 'CZ', 'M'], codes=[], gadgets=[]


## Inspecting Layer 0 Logical Instructions & Instruction Actions

Let's inspect the exact algebraic specification of each instruction in `protocol.layers[0].instruction_set.instructions` and view a slice of the serialized `.qodec` YAML.

In [3]:
# Inspect Layer 0 Instruction Actions & YAML Representation
from _unit import simulation
simulation.ensure_course_state(globals(), stage="baseline")

layer0_instructions = protocol.layers[0].instruction_set.instructions

for name, instr in layer0_instructions.items():
    acts = [str(a) for a in instr.action]
    obs_cnt = instr.observe_count
    flgs = list(instr.flags)
    print(f"Instruction `{name}`:")
    print(f"  inputs={list(instr.inputs)} -> outputs={list(instr.outputs)}")
    print(f"  action={acts}")
    if obs_cnt:
        print(f"  observe_count={obs_cnt}")
    if flgs:
        print(f"  flags={flgs}")

print("\n--- First 35 lines of `protocol.dumps()` (.qodec YAML) ---")
yaml_lines = protocol.dumps().splitlines()
print("\n".join(yaml_lines[:35]))

Instruction `prepare_z_all`:
  inputs=[] -> outputs=[BlockOperand("C4", is_variadic=False)]
  action=['stabilize: [Z_0, Z_1]\n']
Instruction `prepare_x_all`:
  inputs=[] -> outputs=[BlockOperand("C4", is_variadic=False)]
  action=['stabilize: [X_0, X_1]\n']
Instruction `syndrome`:
  inputs=[BlockOperand("C4", is_variadic=False)] -> outputs=[BlockOperand("C4", is_variadic=False)]
  action=[]
Instruction `measure_z_all`:
  inputs=[BlockOperand("C4", is_variadic=False)] -> outputs=[]
  action=['observe: [Z_0, Z_1]\n']
  observe_count=2
Instruction `measure_x_all`:
  inputs=[BlockOperand("C4", is_variadic=False)] -> outputs=[]
  action=['observe: [X_0, X_1]\n']
  observe_count=2
Instruction `cx_all`:
  inputs=[BlockOperand("C4", is_variadic=False), BlockOperand("C4", is_variadic=False)] -> outputs=[BlockOperand("C4", is_variadic=False), BlockOperand("C4", is_variadic=False)]
  action=['clifford:\n  X_0: X_0 X_2\n  X_1: X_1 X_3\n  Z_2: Z_0 Z_2\n  Z_3: Z_1 Z_3\n']

--- First 35 lines of `pro

## Customizing the Physical Instruction Set (`Layer 1`) for Neutral-Atom Hardware (`CZ` & `SWAP`)

When `ec.build_qodec(c4, strategy="bare-css/v1", strict=False)` synthesizes a protocol, it populates `protocol.layers[1].instruction_set` (`physical.isa.yaml`) with the standard 8 physical primitives `{I, H, X, Y, Z, CX, R, M}`.

In real hardware co-design—such as **Magne's neutral-atom architecture**—you will often want to add **custom physical instructions** to Layer 1 so your Layer 0 gadgets can use native hardware primitives directly:
1. **Native Rydberg Blockade Controlled-$Z$ (`CZ`):** Neutral-atom two-qubit gates natively implement `CZ` (`X_0 → X_0 Z_1`, `Z_0 → Z_0`, `X_1 → Z_0 X_1`, `Z_1 → Z_1`).
2. **Optical Tweezer Atom Permutation (`SWAP`):** Movable acousto-optic deflector (AOD) tweezers can physically exchange two atoms (`X_0 → X_1`, `Z_0 → Z_1`, `X_1 → X_0`, `Z_1 → Z_0`).

> **🔬 QEC Researcher Insight — Two-Qubit Gate `SWAP 1 2` vs. Independent Atom Shuttling:**
> Watch what happens when we implement an un-swapped logical Hadamard (`hadamard_all`: `X_0 → Z_0, Z_0 → X_0, X_1 → Z_1, Z_1 → X_1`) on $C_4$ using `H 0 1 2 3` followed by `SWAP 1 2`:
> - `ec.GadgetProfile` verifies that `H 0 1 2 3` + `SWAP 1 2` is **100% algebraically equivalent** to the un-swapped logical Hadamard (`is_equivalent_to(objective) == True`)!
> - However, `ec.GadgetProfile(g).distance()` reports **`d_circuit = 1`** (`witness: X_1 X_2 after call 4`)! Why? Because `GadgetProfile.distance()` treats any 2-input instruction (`SWAP 1 2`) as a two-qubit gate where a *single fault event* can inject a correlated weight-2 error `X_1 X_2` across both participating qubits!
> - That immediately reveals why neutral-atom architects prefer **independent 1-qubit atom transport channels** (or classical qubit-index relabeling in the compiler's Pauli frame tracker) over executing a noisy 2-qubit `SWAP` gate!

In [4]:
# Extend Layer 1 (`physical` ISA) with Native Neutral-Atom `CZ` and `SWAP` Primitives
from _unit import simulation
simulation.ensure_course_state(globals(), stage="baseline")

# Work on a copy of the protocol so we can freely experiment with Layer 1 primitives
na_proto = qodec.Qodec.loads(protocol.dumps())
phys_isa = na_proto.layers[1].instruction_set

# 1. Register native Rydberg `CZ` and optical-tweezer `SWAP` in Layer 1 (`physical` ISA)
phys_isa.instructions["CZ"] = qodec.Instruction(
    mnemonic="CZ",
    description="Neutral-atom Rydberg blockade controlled-Z gate.",
    inputs=[qodec.instructions.BlockOperand("qubit"), qodec.instructions.BlockOperand("qubit")],
    outputs=[qodec.instructions.BlockOperand("qubit"), qodec.instructions.BlockOperand("qubit")],
    action=[qodec.actions.Clifford({"X_0": "X_0 Z_1", "Z_0": "Z_0", "X_1": "Z_0 X_1", "Z_1": "Z_1"})],
)
phys_isa.instructions["SWAP"] = qodec.Instruction(
    mnemonic="SWAP",
    description="Optical tweezer physical atom-pair swap.",
    inputs=[qodec.instructions.BlockOperand("qubit"), qodec.instructions.BlockOperand("qubit")],
    outputs=[qodec.instructions.BlockOperand("qubit"), qodec.instructions.BlockOperand("qubit")],
    action=[qodec.actions.Clifford({"X_0": "X_1", "Z_0": "Z_1", "X_1": "X_0", "Z_1": "Z_0"})],
)

# 2. Define an un-swapped logical Hadamard (`hadamard_all`) in Layer 0 using `H 0 1 2 3` + `SWAP 1 2`
h_all_instr = qodec.Instruction(
    mnemonic="hadamard_all",
    inputs=[qodec.instructions.BlockOperand("C4")],
    outputs=[qodec.instructions.BlockOperand("C4")],
    action=[qodec.actions.Clifford({"X_0": "Z_0", "Z_0": "X_0", "X_1": "Z_1", "Z_1": "X_1"})],
)
na_proto.layers[0].instruction_set.instructions["hadamard_all"] = h_all_instr
h_all_gadget = ec.filled(
    qodec.Gadget(
        implements=h_all_instr,
        inputs=list(na_proto.layers[0].gadgets["syndrome"].inputs),
        outputs=list(na_proto.layers[0].gadgets["syndrome"].outputs),
        circuit=qodec.gadgets.Circuit(
            format="stim",
            source="H 0 1 2 3\nSWAP 1 2\n",
            instruction_set=phys_isa,
        ),
    )
)
h_prof = ec.GadgetProfile(h_all_gadget)

print("Layer 1 physical instructions :", list(phys_isa.instructions.keys()))
print("`hadamard_all` (`H 0 1 2 3` + `SWAP 1 2`) matches objective?", h_prof.action.is_equivalent_to(h_prof.objective))
print("`hadamard_all` synthesized checks:", h_all_gadget.checks)
print("`hadamard_all` circuit distance with 2Q `SWAP 1 2` fault model:", h_prof.distance().value, "| witness:", h_prof.distance().witness)

Layer 1 physical instructions : ['R', 'H', 'CX', 'CZ', 'M', 'SWAP']
`hadamard_all` (`H 0 1 2 3` + `SWAP 1 2`) matches objective? True
`hadamard_all` synthesized checks: ((Reference('out[0].stabilizers[0]'), Reference('in[0].stabilizers[1]')), (Reference('out[0].stabilizers[1]'), Reference('in[0].stabilizers[0]')))
`hadamard_all` circuit distance with 2Q `SWAP 1 2` fault model: 1 | witness: X_1 X_2 after call 4


## Knowledge check

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why did `ec.build_qodec(c4, strategy='bare-css/v1', strict=True)` fail on $C_4$, and why did `strict=False` succeed?</summary>

<div style="padding:0.1em 1em;">

`strict=True` enforces that every synthesized gadget in the protocol has a circuit distance at least equal to the code distance ($d=2$). The `bare-css/v1` strategy uses unflagged single-ancilla syndrome measurement and unflagged preparation, which suffer from weight-2 problematic errors on $C_4$ (`circuit distance = 1 < 2`). `strict=False` permits gadgets whose circuit distance is below the code distance so you can inspect and upgrade them.

</div>

</details></div>

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;In `qodec`, how does an instruction specify state preparation, unitary transformation, and measurement in its `action` list?</summary>

<div style="padding:0.1em 1em;">

State preparation uses `qodec.actions.Stabilize(['Z_0', 'Z_1'])` (`stabilize: [Z_0, Z_1]` in YAML), a Clifford unitary uses `qodec.actions.Clifford({'X_0': 'X_0 X_2', ...})` (`clifford: {...}` in YAML), and destructive measurement uses `qodec.actions.Observe(['Z_0', 'Z_1'])` (`observe: [Z_0, Z_1]` in YAML).

</div>

</details></div>

## Exercise: Count Synthesized Gadgets in a Baseline CSS Protocol

Inspect the baseline two-layer `Qodec` synthesized by `ec.build_qodec(c4, strategy='bare-css/v1', strict=False)` and return the number of gadgets in Layer 0.

In [ ]:
from _unit import exercise, simulation


@exercise
def c4_layer0_gadget_count() -> int:
    ns: dict = {}
    simulation.ensure_course_state(ns, 'baseline')
    protocol = ns['protocol']
    # Return the number of gadgets in Layer 0 of protocol
    return ...

**Hint**

Access `len(protocol.layers[0].gadgets)`.

In [ ]:
from _unit import exercise, simulation


@exercise
def c4_layer0_gadget_count() -> int:
    ns: dict = {}
    simulation.ensure_course_state(ns, 'baseline')
    protocol = ns['protocol']
    return len(protocol.layers[0].gadgets)

`ec.build_qodec(c4, strategy='bare-css/v1', strict=False)` synthesizes `6` baseline Layer 0 gadgets: `prepare_z_all`, `prepare_x_all`, `measure_z_all`, `measure_x_all`, `syndrome`, and `cx_all`.

---

### 🤖 COPILOT INSTRUCTOR BRIEF — Module 2
- **Learning Objectives:**
  1. Synthesize a baseline 2-layer `qodec.Qodec` from `c4` using `ec.build_qodec(c4, strategy="bare-css/v1", strict=False)`.
  2. Navigate `protocol.layers[0]` and `protocol.layers[1]`, inspecting `.instruction_set.instructions`, `.codes`, and `.gadgets`.
  3. Read and write `qodec` instruction actions (`Stabilize`, `Clifford`, `Pauli`, `Observe`) and serialize/deserialize via `protocol.dumps()` / `qodec.Qodec.loads()`.
- **Why `strict=False`?** By default (`strict=True`), `ec.build_qodec` requires every synthesized gadget to achieve circuit distance equal to the code distance ($d=2$). Because `bare-css/v1` uses bare ancillas whose problematic errors reduce `syndrome` and `prepare_*` to circuit distance 1 on $C_4$, `strict=True` raises an error! Passing `strict=False` lets us inspect the baseline $d_{\text{circuit}}=1$ protocol first and upgrade it in Module 6.
- **Socratic Prompt:** Have the learner compare `strict=True` vs `strict=False` in Cell 1 below to see QDK-EC's built-in fault-tolerance guardrails in action.

<div style="display:flex;justify-content:space-between;align-items:stretch;gap:12px;margin-top:8px"><a href="../01-stabilizer-codes/stabilizer_codes.workbook.ipynb" title="Previous unit: Define & Profile a Stabilizer Code" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-secondaryBackground, transparent);color:var(--vscode-button-secondaryForeground, inherit);border-color:var(--vscode-widget-border, #8884)"><span aria-hidden="true">&#8249;</span><span><span style="display:block;font-size:11px;opacity:.75">Previous unit</span><span style="display:block;font-weight:600">Define & Profile a Stabilizer Code</span></span></a><a href="../03-gadget-anatomy/gadget_anatomy.workbook.ipynb" title="Next unit: Inspect Gadget Circuits, Checks & Readouts" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-background, #005fb8);color:var(--vscode-button-foreground, #ffffff);border-color:var(--vscode-button-background, #005fb8)"><span style="text-align:right"><span style="display:block;font-size:11px;opacity:.75">Next unit</span><span style="display:block;font-weight:600">Inspect Gadget Circuits, Checks & Readouts</span></span><span aria-hidden="true">&#8250;</span></a></div>